In [1]:
import sys
import os
from pathlib import Path
sys.path.append(os.path.abspath("/mount/NAS-workspace-portal/eeg2025-Vistec/"))

from eegkit.utils.system.logging_utils import configure_logging, silence_console_logs
from eegkit.models.subject_model import EEGSubjectModel
from eegkit.controller.eeg_controller import EEGController
from eegkit.models.dtos import TaskDTO, SubjectFilterDTO
from eegkit.models.dtos import EpochParamsDTO, EvokedParamsDTO
from eegkit.utils.channels import prepare_channels

configure_logging('INFO')
silence_console_logs('WARNING')

import torch
import torch.nn as nn

In [30]:
DATA_DIR = Path(os.getenv("DATA_DIR") or os.getenv("EEG_DATA_DIR") or "")
if not DATA_DIR or not DATA_DIR.exists():
    raise FileNotFoundError("Set DATA_DIR to your BIDS root containing cmi_bids_R*/ directories.")

subject_model = EEGSubjectModel(DATA_DIR)
controller = EEGController(subject_model)

In [3]:
# Parameters: subjects, channels, and epoch window
N_SUBJECTS = 500  # use first N subjects
CHANNELS = "69-76,81-83,88,89"  # explicit posterior channels

# Epoch feature extraction params ([-2, 0] s)
params = EpochParamsDTO(
    tmin=-2.0,
    tmax=0.0,
    stimulus=[None],
    channels=CHANNELS,
    notch=None,
    uv_min=None,
    uv_max=None,
    clean_flatline_sec=None,
    clean_hf_noise_sd_max=None,
    clean_corr_min=None,
    clean_asr_max_std=None,
    clean_power_min_sd=None,
    clean_power_max_sd=None,
    clean_max_outbound_pct=None,
    clean_window_sec=None,
)
print(f"Subjects: {N_SUBJECTS}, Channels: {CHANNELS}")

Subjects: 500, Channels: 69-76,81-83,88,89


In [32]:
# Get epochs for all subjects and build per-epoch response labels (0=miss, 1=hit).
import numpy as np
import pandas as pd

group_dto = SubjectFilterDTO(task='contrastChangeDetection', subject_limit=N_SUBJECTS)
all_task_dtos = subject_model.get_filter_subjects_dto(group_dto)

def build_ccd_trial_labels(events_df: pd.DataFrame) -> np.ndarray:
    """Build one label per CCD trial start: 1=hit(smiley_face), 0=miss/non-hit."""
    if events_df is None or events_df.empty:
        return np.asarray([], dtype=np.int64)

    df = events_df.copy()
    if "onset" not in df.columns or "value" not in df.columns:
        return np.asarray([], dtype=np.int64)

    df["onset"] = pd.to_numeric(df["onset"], errors="coerce")
    df = df.dropna(subset=["onset"]).sort_values("onset").reset_index(drop=True)

    trial_start_idx = np.where(df["value"].eq("contrastTrial_start").values)[0]
    if trial_start_idx.size == 0:
        return np.asarray([], dtype=np.int64)

    labels = []
    onsets = df["onset"].to_numpy()
    values = df["value"].astype(str).to_numpy()

    for i, start_idx in enumerate(trial_start_idx):
        t_start = float(onsets[start_idx])
        t_end = float(onsets[trial_start_idx[i + 1]]) if i + 1 < len(trial_start_idx) else np.inf

        in_window = (onsets > t_start) & (onsets < t_end)
        press_mask = np.isin(values, ["left_buttonPress", "right_buttonPress"]) & in_window
        press_idx = np.where(press_mask)[0]

        if press_idx.size == 0:
            labels.append(0)
            continue

        first_press = int(press_idx[0])
        fb = None
        if "feedback" in df.columns and pd.notna(df.iloc[first_press].get("feedback")):
            fb = str(df.iloc[first_press]["feedback"]).strip()

        labels.append(1 if fb == "smiley_face" else 0)

    return np.asarray(labels, dtype=np.int64)

# epoch_list contains epoch objects per run; responds_list is flattened per-epoch labels across all runs.
epoch_list = []
responds_list = []
subject_ids = []

skipped = 0
for dto in all_task_dtos:
    task_model = subject_model.get_task(dto)
    events_df = task_model.get_event()
    trial_labels = build_ccd_trial_labels(events_df)
    if trial_labels.size == 0:
        skipped += 1
        continue

    epoch, _ = controller.get_epochs(dto, params)
    if epoch is None:
        skipped += 1
        continue

    epoch = prepare_channels(epoch, params)

    # epoch.selection holds original event indices; convert to in-epoch positions safely.
    selection = np.asarray(getattr(epoch, "selection", np.arange(len(epoch), dtype=int)), dtype=int)
    valid_pos = np.where((selection >= 0) & (selection < len(trial_labels)))[0]
    if valid_pos.size == 0:
        skipped += 1
        continue

    epoch_kept = epoch[valid_pos]
    labels_kept = trial_labels[selection[valid_pos]]

    epoch_list.append(epoch_kept)
    responds_list.extend(labels_kept.tolist())
    subject_ids.extend([dto.subject] * len(epoch_kept))

responds_list = np.asarray(responds_list, dtype=np.int64)
subject_ids = np.asarray(subject_ids)

print(f"Runs considered: {len(all_task_dtos)}")
print(f"Runs skipped: {skipped}")
print(f"Epoch blocks: {len(epoch_list)}")
print(f"Labels shape: {responds_list.shape}")

Runs considered: 287
Runs skipped: 0
Epoch blocks: 287
Labels shape: (8623,)


In [33]:
# Sanity check: epoch blocks, labels, and subject IDs alignment
import numpy as np
import pandas as pd

required_vars = ["epoch_list", "responds_list", "subject_ids"]
missing = [v for v in required_vars if v not in globals()]
if missing:
    raise RuntimeError(f"Missing required variables from previous cell: {missing}")

# Convert to arrays for robust checks
responds_arr = np.asarray(responds_list)
subject_ids_arr = np.asarray(subject_ids)

block_lengths = np.array([len(ep) for ep in epoch_list], dtype=int)
total_epochs_from_blocks = int(block_lengths.sum())

print("=== Alignment Summary ===")
print(f"Epoch blocks: {len(epoch_list)}")
print(f"Total epochs from epoch_list: {total_epochs_from_blocks}")
print(f"Labels length: {len(responds_arr)}")
print(f"Subject IDs length: {len(subject_ids_arr)}")

ok = True
if total_epochs_from_blocks != len(responds_arr):
    ok = False
    print("[FAIL] Total epochs != labels length")
if len(responds_arr) != len(subject_ids_arr):
    ok = False
    print("[FAIL] Labels length != subject_ids length")

# Per-block cumulative index check
start = 0
for i, n in enumerate(block_lengths):
    stop = start + n
    if stop > len(responds_arr):
        ok = False
        print(f"[FAIL] Block {i}: slice out of range (start={start}, stop={stop})")
        break
    start = stop

if start != len(responds_arr):
    ok = False
    print(f"[FAIL] Not all labels consumed by block lengths (consumed={start}, labels={len(responds_arr)})")

# Label diagnostics
if len(responds_arr) > 0:
    uniq, cnt = np.unique(responds_arr, return_counts=True)
    label_stats = pd.DataFrame({"label": uniq, "count": cnt, "ratio": cnt / cnt.sum()})
    print("\n=== Label Distribution ===")
    display(label_stats)
else:
    print("[WARN] responds_list is empty")

# Subject diagnostics
if len(subject_ids_arr) > 0:
    subj_unique, subj_cnt = np.unique(subject_ids_arr, return_counts=True)
    subj_df = pd.DataFrame({"subject": subj_unique, "n_epochs": subj_cnt}).sort_values("n_epochs", ascending=False)
    print("\n=== Subject Epoch Count (top 10) ===")
    display(subj_df.head(10))
else:
    print("[WARN] subject_ids is empty")

if ok:
    print("\n[PASS] Epochs, labels, and subject IDs are aligned.")
else:
    raise AssertionError("Sanity check failed: epoch-label alignment mismatch")

=== Alignment Summary ===
Epoch blocks: 287
Total epochs from epoch_list: 8623
Labels length: 8623
Subject IDs length: 8623

=== Label Distribution ===


,label,count,ratio
0,0,2785,0.322973
1,1,5838,0.677027



=== Subject Epoch Count (top 10) ===


,subject,n_epochs
36,sub-NDARGX001CB1,126
47,sub-NDARLH979WFX,125
43,sub-NDARKD134TCX,117
23,sub-NDAREN519BLJ,116
96,sub-NDARZC499NVX,115
16,sub-NDARCZ947WU5,100
6,sub-NDARBH024NH2,100
5,sub-NDARBD879MBX,100
19,sub-NDARDU986RBM,100
11,sub-NDARCJ594BWQ,100



[PASS] Epochs, labels, and subject IDs are aligned.


In [5]:
# Save/Load cached dataset (X, y, groups) for faster reruns
from pathlib import Path
from dataclasses import asdict, is_dataclass
import hashlib
import json
import numpy as np

CACHE_DIR = Path("/mount/NAS-workspace-portal/eeg2025-Vistec/models/data")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def _stable_json_default(obj):
    """Convert non-JSON objects to stable strings for cache-key hashing."""
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return str(obj)

def build_cache_key(params_obj, n_subjects: int) -> str:
    """Build cache key from N_SUBJECTS and full params content."""
    if is_dataclass(params_obj):
        params_dict = asdict(params_obj)
    elif hasattr(params_obj, "__dict__"):
        params_dict = dict(params_obj.__dict__)
    else:
        params_dict = {"params_repr": str(params_obj)}

    params_json = json.dumps(
        params_dict,
        sort_keys=True,
        default=_stable_json_default,
        separators=(",", ":"),
    )
    params_hash = hashlib.sha256(params_json.encode("utf-8")).hexdigest()[:12]
    return f"ccd_eegnet_n{int(n_subjects)}_p{params_hash}"

cache_key = build_cache_key(params, N_SUBJECTS)
X_path = CACHE_DIR / f"X_{cache_key}.npy"
y_path = CACHE_DIR / f"y_{cache_key}.npy"
g_path = CACHE_DIR / f"groups_{cache_key}.npy"

def save_dataset_cache(X_arr, y_arr, groups_arr):
    """Save dataset arrays to disk for reuse."""
    np.save(X_path, X_arr)
    np.save(y_path, y_arr)
    np.save(g_path, np.asarray(groups_arr).astype(str))
    print("Saved:")
    print(f"  key={cache_key}")
    print(f"  {X_path}")
    print(f"  {y_path}")
    print(f"  {g_path}")

def load_dataset_cache():
    """Load dataset arrays from disk cache if present."""
    if not (X_path.exists() and y_path.exists() and g_path.exists()):
        return None, None, None
    X_arr = np.load(X_path)
    y_arr = np.load(y_path)
    groups_arr = np.load(g_path)
    print("Loaded:")
    print(f"  key={cache_key}")
    print(f"  X: {X_arr.shape} from {X_path.name}")
    print(f"  y: {y_arr.shape} from {y_path.name}")
    print(f"  groups: {groups_arr.shape} from {g_path.name}")
    return X_arr, y_arr, groups_arr

# Fully automatic behavior (no manual mode):
if "X" in globals() and "y" in globals() and "groups" in globals():
    save_dataset_cache(X, y, groups)
else:
    X_loaded, y_loaded, g_loaded = load_dataset_cache()
    if X_loaded is not None:
        X, y, groups = X_loaded, y_loaded, g_loaded
    else:
        print("No cache found for current (N_SUBJECTS, params) key.")
        print("Run dataset build cell first, then rerun this cell to auto-save.")
        print(f"Expected key: {cache_key}")

No cache found for current (N_SUBJECTS, params) key.
Run dataset build cell first, then rerun this cell to auto-save.
Expected key: ccd_eegnet_n500_p6b3c980fa74c


In [34]:
# Build subject-independent dataset for EEGNet
import numpy as np

if len(epoch_list) == 0:
    raise RuntimeError("epoch_list is empty. Run the previous data-prep cell first.")

# Concatenate all epoch blocks: (S*N, C, T)
X_blocks = [ep.get_data().astype(np.float32, copy=False) for ep in epoch_list]
X_ct = np.concatenate(X_blocks, axis=0)  # (N_total, C, T)

# Labels and subject groups aligned from previous steps
y = np.asarray(responds_list, dtype=np.int64)  # (N_total,)
groups = np.asarray(subject_ids)               # (N_total,) subject IDs for subject-independent split

if X_ct.shape[0] != len(y):
    raise AssertionError(f"Mismatch: X rows ({X_ct.shape[0]}) != y length ({len(y)})")
if len(y) != len(groups):
    raise AssertionError(f"Mismatch: y length ({len(y)}) != groups length ({len(groups)})")

# EEGNet input expects (N, D, C, T). Use D=1 for single feature depth.
X = np.expand_dims(X_ct, axis=1)  # (N_total, 1, C, T)

print("=== Dataset Built ===")
print(f"X shape (N,D,C,T): {X.shape}")
print(f"y shape (N,): {y.shape}")
print(f"Unique labels: {np.unique(y)}")
print(f"Unique subjects: {len(np.unique(groups))}")

=== Dataset Built ===
X shape (N,D,C,T): (8623, 1, 13, 201)
y shape (N,): (8623,)
Unique labels: [0 1]
Unique subjects: 100


In [35]:

class Conv2dWithConstraint(nn.Conv2d):
    """Custom Conv2d to implement max_norm constraints."""
    def __init__(self, *args, max_norm=1.0, **kwargs):
        self.max_norm = max_norm
        super(Conv2dWithConstraint, self).__init__(*args, **kwargs)

    def forward(self, x):
        self.weight.data = torch.renorm(
            self.weight.data, p=2, dim=0, maxnorm=self.max_norm
        )
        return super(Conv2dWithConstraint, self).forward(x)

class EEGNet(nn.Module):
    def __init__(self, nb_classes, Chans=64, Samples=128, 
                 dropoutRate=0.5, kernLength=64, F1=8, 
                 D=2, F2=16, norm_rate=0.25, dropoutType='Dropout'):
        super(EEGNet, self).__init__()
        
        self.F1 = F1
        self.D = D
        self.F2 = F2
        
        # Select Dropout Layer
        if dropoutType == 'SpatialDropout2D':
            self.dropout = nn.Dropout2d(p=dropoutRate)
        else:
            self.dropout = nn.Dropout(p=dropoutRate)

        # Block 1
        self.block1 = nn.Sequential(
            # Temporal Convolution
            nn.Conv2d(1, F1, (1, kernLength), padding=(0, kernLength // 2), bias=False),
            nn.BatchNorm2d(F1),
            # Depthwise Convolution (Spatial Filter)
            Conv2dWithConstraint(F1, F1 * D, (Chans, 1), groups=F1, bias=False, max_norm=1.0),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            self.dropout
        )

        # Block 2 (Separable Convolution)
        self.block2 = nn.Sequential(
            # Depthwise
            nn.Conv2d(F1 * D, F1 * D, (1, 16), padding=(0, 8), groups=F1 * D, bias=False),
            # Pointwise
            nn.Conv2d(F1 * D, F2, (1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            self.dropout
        )

        # Final Classifier
        # Note: Calculating flatten size based on pooling
        self.flatten_size = F2 * (Samples // 4 // 8) 
        self.classifier = nn.Sequential(
            nn.Linear(self.flatten_size, nb_classes),
            nn.Softmax(dim=1)
        )

    def forward(self, x):
        # Input shape: (Batch, 1, Chans, Samples)
        x = self.block1(x)
        x = self.block2(x)
        x = x.view(x.size(0), -1) # Flatten
        x = self.classifier(x)
        return x

In [36]:
# Subject-independent 50/25/25 split and DataLoader build
from sklearn.model_selection import GroupShuffleSplit
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import torch

if "X" not in globals() or "y" not in globals() or "groups" not in globals():
    raise RuntimeError("Run dataset build cell first to create X, y, groups.")

# Force CPU for now (user request).
device = torch.device("cpu")

# Split 1: 50% train, 50% temp (subject-independent)
gss_1 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
train_idx, temp_idx = next(gss_1.split(X, y, groups=groups))

X_train = X[train_idx].astype(np.float32, copy=False)
y_train = y[train_idx].astype(np.int64, copy=False)
groups_train = groups[train_idx]

X_temp = X[temp_idx].astype(np.float32, copy=False)
y_temp = y[temp_idx].astype(np.int64, copy=False)
groups_temp = groups[temp_idx]

# Split 2: 50% of temp for valid, 50% for test => overall 25%/25%
gss_2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=43)
valid_idx_rel, test_idx_rel = next(gss_2.split(X_temp, y_temp, groups=groups_temp))

X_valid = X_temp[valid_idx_rel]
y_valid = y_temp[valid_idx_rel]
groups_valid = groups_temp[valid_idx_rel]

X_test = X_temp[test_idx_rel]
y_test = y_temp[test_idx_rel]
groups_test = groups_temp[test_idx_rel]

# Per-channel normalization using train set only to avoid leakage
mu = X_train.mean(axis=(0, 3), keepdims=True)
sd = X_train.std(axis=(0, 3), keepdims=True) + 1e-6
X_train = (X_train - mu) / sd
X_valid = (X_valid - mu) / sd
X_test = (X_test - mu) / sd

X_train_t = torch.from_numpy(X_train)
X_valid_t = torch.from_numpy(X_valid)
X_test_t = torch.from_numpy(X_test)
y_train_t = torch.from_numpy(y_train)
y_valid_t = torch.from_numpy(y_valid)
y_test_t = torch.from_numpy(y_test)

train_ds = TensorDataset(X_train_t, y_train_t)
valid_ds = TensorDataset(X_valid_t, y_valid_t)
test_ds = TensorDataset(X_test_t, y_test_t)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=256, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=0)

# Class weights from train split only
classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight_tensor = torch.tensor(weights, dtype=torch.float32, device=device)

print("=== Subject-Independent Split (50/25/25) ===")
print(f"Train samples: {len(X_train)}, Valid samples: {len(X_valid)}, Test samples: {len(X_test)}")
print(
    f"Train subjects: {len(np.unique(groups_train))}, "
    f"Valid subjects: {len(np.unique(groups_valid))}, "
    f"Test subjects: {len(np.unique(groups_test))}"
)
ov_tv = len(set(groups_train).intersection(set(groups_valid)))
ov_tt = len(set(groups_train).intersection(set(groups_test)))
ov_vt = len(set(groups_valid).intersection(set(groups_test)))
print(f"Subject overlaps (train-valid/train-test/valid-test): {ov_tv}/{ov_tt}/{ov_vt}")
print(f"Train label counts: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"Valid label counts: {dict(zip(*np.unique(y_valid, return_counts=True)))}")
print(f"Test label counts: {dict(zip(*np.unique(y_test, return_counts=True)))}")
print(f"Device: {device}")

=== Subject-Independent Split (50/25/25) ===
Train samples: 4283, Valid samples: 2190, Test samples: 2150
Train subjects: 50, Valid subjects: 25, Test subjects: 25
Subject overlaps (train-valid/train-test/valid-test): 0/0/0
Train label counts: {np.int64(0): np.int64(1490), np.int64(1): np.int64(2793)}
Valid label counts: {np.int64(0): np.int64(669), np.int64(1): np.int64(1521)}
Test label counts: {np.int64(0): np.int64(626), np.int64(1): np.int64(1524)}
Device: cpu


In [ ]:
# Train and evaluate EEGNet (CPU, best model by validation BACC)
import torch
import numpy as np
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix

# Build model with data-driven channel/time dimensions
model = EEGNet(
    nb_classes=2,
    Chans=X_train.shape[2],
    Samples=X_train.shape[3],
    dropoutRate=0.5,
    kernLength=64,
    F1=8,
    D=2,
    F2=16,
    dropoutType='Dropout',
).to(device)

# CrossEntropyLoss expects logits; remove final Softmax if present.
if isinstance(model.classifier, torch.nn.Sequential) and isinstance(model.classifier[-1], torch.nn.Softmax):
    model.classifier = torch.nn.Sequential(*list(model.classifier.children())[:-1]).to(device)

criterion = torch.nn.CrossEntropyLoss(weight=class_weight_tensor)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

EPOCHS = 8
best_state = None
best_val_bacc = -np.inf
history = []

epoch_bar = tqdm(range(1, EPOCHS + 1), desc="Epochs", leave=True)
for epoch in epoch_bar:
    model.train()
    train_loss = 0.0
    train_true = []
    train_pred = []

    train_bar = tqdm(train_loader, desc=f"Train {epoch:02d}", leave=False)
    for xb, yb in train_bar:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * xb.size(0)
        pred = torch.argmax(logits, dim=1)
        train_true.append(yb.detach().cpu().numpy())
        train_pred.append(pred.detach().cpu().numpy())

        train_bar.set_postfix(loss=float(loss.item()))

    train_true = np.concatenate(train_true)
    train_pred = np.concatenate(train_pred)
    train_loss /= len(train_loader.dataset)
    train_acc = accuracy_score(train_true, train_pred)
    train_bacc = balanced_accuracy_score(train_true, train_pred)

    model.eval()
    val_loss = 0.0
    val_true = []
    val_pred = []
    valid_bar = tqdm(valid_loader, desc=f"Valid {epoch:02d}", leave=False)
    with torch.no_grad():
        for xb, yb in valid_bar:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            val_loss += loss.item() * xb.size(0)
            pred = torch.argmax(logits, dim=1)
            val_true.append(yb.detach().cpu().numpy())
            val_pred.append(pred.detach().cpu().numpy())

            valid_bar.set_postfix(loss=float(loss.item()))

    val_true = np.concatenate(val_true)
    val_pred = np.concatenate(val_pred)
    val_loss /= len(valid_loader.dataset)
    val_acc = accuracy_score(val_true, val_pred)
    val_bacc = balanced_accuracy_score(val_true, val_pred)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "train_bacc": train_bacc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_bacc": val_bacc,
    })

    if val_bacc > best_val_bacc:
        best_val_bacc = val_bacc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    epoch_bar.set_postfix(
        train_loss=f"{train_loss:.4f}",
        train_bacc=f"{train_bacc:.4f}",
        valid_loss=f"{val_loss:.4f}",
        valid_bacc=f"{val_bacc:.4f}",
    )

if best_state is not None:
    model.load_state_dict(best_state)

# Final test evaluation
model.eval()
all_true, all_pred = [], []
test_bar = tqdm(test_loader, desc="Test", leave=False)
with torch.no_grad():
    for xb, yb in test_bar:
        xb = xb.to(device)
        logits = model(xb)
        pred = torch.argmax(logits, dim=1).cpu().numpy()
        all_pred.append(pred)
        all_true.append(yb.numpy())

all_true = np.concatenate(all_true)
all_pred = np.concatenate(all_pred)

print("\n=== Final Test Metrics (Best Valid BACC) ===")
print(f"Accuracy: {accuracy_score(all_true, all_pred):.4f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(all_true, all_pred):.4f}")
print("Confusion Matrix:")
print(confusion_matrix(all_true, all_pred))
print("\nClassification Report:")
print(classification_report(all_true, all_pred, digits=4))

Epochs:   0%|          | 0/8 [00:00<?, ?it/s]

Train 01:   0%|          | 0/67 [00:00<?, ?it/s]

Valid 01:   0%|          | 0/9 [00:00<?, ?it/s]

Train 02:   0%|          | 0/67 [00:00<?, ?it/s]

Valid 02:   0%|          | 0/9 [00:00<?, ?it/s]

...

Valid 05:   0%|          | 0/9 [00:00<?, ?it/s]

Train 06:   0%|          | 0/67 [00:00<?, ?it/s]

Valid 06:   0%|          | 0/9 [00:00<?, ?it/s]

Train 07:   0%|          | 0/67 [00:00<?, ?it/s]

Valid 07:   0%|          | 0/9 [00:00<?, ?it/s]

In [ ]:
class_weights = class_weights(y_train, device)
                loss_fn = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.2)

def class_weights(y, device):
    if isinstance(y, torch.Tensor):
        y = y.cpu().numpy()
    _labels = y.astype(int)
    class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(_labels), y=_labels)
    print('class weights:', class_weights)
    return torch.from_numpy(class_weights).float().to(device)